# GAE on Colab: camera-controlled video, images and 3D from one geometry-native latent

<a href="https://colab.research.google.com/github/Cortex78/GAE-GeometricAutoEncoder/blob/main/notebooks/GAE_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook installs and runs the inference code in [this repository](https://github.com/Cortex78/GAE-GeometricAutoEncoder) ([paper](https://arxiv.org/abs/2609.24981), [project page](https://jiah-cloud.github.io/GAE.github.io/), [weights](https://huggingface.co/TencentARC/GAE-D64-1B)). GAE decodes RGB **and** geometry from the same latent, so every result comes with depth and a 3D point cloud.

| Part | Task | Input | Output |
| --- | --- | --- | --- |
| **A** | Image → camera-controlled video | one image + a text prompt + a camera path | generated video, depth video, 3D point cloud, camera-path plot |
| **B** | Text → image | a prompt | image, depth map, 3D point cloud |
| **C** | Codec reconstruction | a video or an image | reconstructed RGB and depth, 3D point cloud |

Run **Setup** first, then any part.

**Before you start:** choose *Runtime → Change runtime type → A100 GPU*. The notebook is set up for an A100.

**Everything runs on the GPU:**
* The install cell checks that PyTorch can use CUDA (bf16 throughput, FlashAttention) and that xFormers attention runs on the GPU.
* Every run reports the peak GPU memory and utilisation it reached.

**Memory:**
* The notebook puts no limit on the number of views or frames.
* If a run does not fit in GPU memory, Parts A and C retry automatically with about 20% fewer views or frames each time. On synthetic camera paths Part A keeps every frame and generates the clip in shorter chunks instead.
* Loading the models takes a lot of system RAM; A100 runtimes have enough.

**Downloads:**
* About 13 GB on first use: the GAE-64 weights, plus DA3-GIANT and Qwen3-0.6B from the Hugging Face Hub.
* The optional automatic scene description for your own images adds about 16 GB (Qwen2.5-VL-7B).

> **License:** GAE is released by Tencent under the [GAE license terms](https://github.com/Cortex78/GAE-GeometricAutoEncoder/blob/main/LICENSE.txt), **for academic use only**; non-academic, commercial or production use is not permitted. Third-party components keep their own licenses.

## 0 · Setup

In [ ]:
#@title 0.1 · Check the runtime (GPU, RAM, disk)
import os, re, shutil, subprocess, sys
from pathlib import Path

query = None
if shutil.which('nvidia-smi'):
    query = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap,driver_version',
                            '--format=csv,noheader,nounits'], capture_output=True, text=True)
if query is None or query.returncode != 0 or not query.stdout.strip():
    raise SystemExit('No NVIDIA GPU found. Choose Runtime > Change runtime type > A100 GPU, then run this cell again.')
gpu_name, vram_mib, compute_cap, driver = [s.strip() for s in query.stdout.strip().splitlines()[0].split(',')]
GPU_NAME, VRAM_GB, COMPUTE_CAP = gpu_name, float(vram_mib) / 1024, float(compute_cap)
# The newest CUDA runtime this driver supports, from the nvidia-smi banner; it picks the PyTorch build.
cuda_banner = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
DRIVER_CUDA = (int(cuda_banner[1]), int(cuda_banner[2])) if cuda_banner else (0, 0)
RAM_GB = int(Path('/proc/meminfo').read_text().split()[1]) / 1024**2
BASE = Path('/content') if Path('/content').is_dir() else Path.cwd()
DISK_FREE_GB = shutil.disk_usage(BASE).free / 1024**3

print(f'GPU : {GPU_NAME}, {VRAM_GB:.0f} GB, compute capability {COMPUTE_CAP}, '
      f'driver {driver} (CUDA {DRIVER_CUDA[0]}.{DRIVER_CUDA[1]})')
print(f'RAM : {RAM_GB:.1f} GB   disk free: {DISK_FREE_GB:.0f} GB   Python {sys.version.split()[0]}')
if 'A100' in GPU_NAME:
    print('A100 detected.')
else:
    print('Note: this notebook is set up for an A100 (Runtime > Change runtime type > A100 GPU).')
if COMPUTE_CAP < 8.0:
    print('Warning: GAE runs in bfloat16, which this GPU does not support natively; expect it to be slow or fail.')
if VRAM_GB < 30:
    print('Note: with this little GPU memory, Part A will likely fall back to fewer views per pass.')
if RAM_GB < 24:
    print('Warning: loading the models takes a lot of system RAM; use an A100 or High-RAM runtime.')
if DISK_FREE_GB < 40:
    print('Warning: the environment, weights and results need about 35 GB of free disk.')

In [ ]:
#@title 0.2 · Get the code
REPO_URL = 'https://github.com/Cortex78/GAE-GeometricAutoEncoder'  #@param {type:"string"}
BRANCH = 'main'  #@param {type:"string"}
#@markdown If this branch does not exist, the repository's default branch is cloned instead.

REPO = BASE / 'GAE-GeometricAutoEncoder'
if (REPO / '.git').is_dir():
    print(f'Reusing the existing clone at {REPO} (delete it to clone a different branch).')
elif subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)]).returncode:
    print(f'Branch {BRANCH} was not found; cloning the default branch.')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
print(subprocess.run(['git', '-C', str(REPO), 'log', '-1', '--format=%h %d %s (%cd)'],
                     capture_output=True, text=True).stdout)

In [ ]:
#@title 0.3 · Helper functions (run once; expand to read the code) { display-mode: "form" }
import codecs, html, shlex, socket, threading, time, urllib.parse, zipfile
import numpy as np
from PIL import Image, ImageOps
from IPython.display import HTML, display

IN_COLAB = 'google.colab' in sys.modules
ENVS, OUTPUTS = BASE / 'envs', BASE / 'outputs'
GAE_PY = ENVS / 'gae' / 'bin' / 'python'
CKPTS = REPO / 'ckpts'  # generate_t2i.py always reads <repo>/ckpts/latent_stats_gae_64.pt, so the weights live here
HF_HOME = BASE / 'hf'  # one Hugging Face cache for every script (they otherwise split it between /tmp and ~/.cache)
HF_REPO = 'TencentARC/GAE-D64-1B'
HUB_MODELS = ['depth-anything/DA3-GIANT-1.1', 'Qwen/Qwen3-0.6B']  # fetched at run time by the scripts
CAPTION_MODEL = 'Qwen/Qwen2.5-VL-7B-Instruct'
DEFAULT_POSES = REPO / 'examples' / 'scenes' / 'forest_lake_trail_poses.npz'  # the app's default path for uploads
OUTPUTS.mkdir(parents=True, exist_ok=True)
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
VIDEO_EXTS = {'.mp4', '.mov', '.m4v', '.webm', '.mkv', '.avi'}
TRAINED_VIEWS = 81  # the flow model's training clip; longer synthetic clips are generated in windows of this size

def env_with(*bin_dirs, **extra):
    """Environment overrides that put the given directories first on PATH."""
    return {'PATH': os.pathsep.join([*map(str, bin_dirs), os.environ['PATH']]), **extra}


# Run inside an environment: fail unless PyTorch can compute on the GPU, and show what it gets.
CUDA_CHECK = r"""
import time, torch, torch.nn.functional as F
if not torch.cuda.is_available():
    raise SystemExit(f"PyTorch {torch.__version__} (CUDA {torch.version.cuda}) cannot use the GPU. Check the "
                     "runtime type; with an old driver, rerun the install cell with TORCH_CUDA = cu121.")
props = torch.cuda.get_device_properties(0)
x = torch.randn(8192, 8192, device="cuda", dtype=torch.bfloat16)
x @ x
torch.cuda.synchronize()
start = time.time()
for _ in range(5):
    x @ x
torch.cuda.synchronize()
tflops = 5 * 2 * 8192**3 / (time.time() - start) / 1e12
try:
    from torch.nn.attention import SDPBackend, sdpa_kernel
    q = torch.randn(1, 8, 2048, 64, device="cuda", dtype=torch.bfloat16)
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        F.scaled_dot_product_attention(q, q, q)
    flash = "yes"
except RuntimeError:
    flash = "no"
print(f"CUDA OK: torch {torch.__version__} (CUDA {torch.version.cuda}) on {props.name}, "
      f"{props.total_memory / 1024**3:.0f} GB, compute {props.major}.{props.minor}; "
      f"bf16 matmul {tflops:.0f} TFLOPS; FlashAttention: {flash}")
"""


def check_cuda(python):
    sh(f'{shlex.quote(str(python))} -c {shlex.quote(CUDA_CHECK)}')


class GpuMonitor:
    """Sample nvidia-smi while a run is in progress and report how much of the GPU it used."""

    def __init__(self, interval=0.25):
        self.interval, self.idle_gb, self.peak_gb, self.peak_util = interval, 0.0, 0.0, 0.0

    @staticmethod
    def sample():
        out = subprocess.run(['nvidia-smi', '-i', '0', '--query-gpu=memory.used,utilization.gpu',
                              '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout
        used, util = out.strip().splitlines()[0].split(',')
        return float(used) / 1024, float(util)

    def __enter__(self):
        try:
            self.idle_gb = self.sample()[0]
        except (ValueError, IndexError, OSError):
            pass
        self.done = threading.Event()

        def loop():
            while True:
                try:
                    used, util = self.sample()
                    self.peak_gb, self.peak_util = max(self.peak_gb, used), max(self.peak_util, util)
                except (ValueError, IndexError, OSError):
                    pass
                if self.done.wait(self.interval):
                    break

        self.thread = threading.Thread(target=loop, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, *exc):
        self.done.set()
        self.thread.join()
        print(f'GPU usage: {GPU_NAME}, peak memory {self.peak_gb:.1f} of {VRAM_GB:.0f} GB, '
              f'peak utilisation {self.peak_util:.0f}%')
        if exc[0] is None and self.peak_gb - self.idle_gb < 1:
            print('WARNING: this run barely used the GPU. Rerun the install cell and check its "CUDA OK" line.')


class CommandError(RuntimeError):
    def __init__(self, code, tail):
        low = tail.lower()
        self.code, self.oom = code, 'out of memory' in low
        if code in (-9, 137):
            hint = ('the process was killed, most likely for lack of system RAM while loading the models; '
                    'use an A100 or High-RAM runtime, or restart the runtime to free memory.')
        elif 'out of memory' in low:
            hint = 'the GPU ran out of memory. Set ON_OOM to retry with fewer views or frames, or ask for fewer.'
        elif 'no space left' in low:
            hint = f'the disk is full. Delete old results in {OUTPUTS} or start a fresh runtime.'
        else:
            hint = 'see the output above.'
        super().__init__(f'Command failed with exit code {code}: {hint}')


def sh(cmd, cwd=None, env=None, show=None):
    """Run a bash command, stream its output, and raise CommandError if it fails."""
    print(f'$ {show or cmd}', flush=True)
    proc = subprocess.Popen(['bash', '-c', 'set -eo pipefail\n' + cmd], cwd=cwd,
                            env={**os.environ, **(env or {})},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    decoder, tail = codecs.getincrementaldecoder('utf-8')('replace'), ''
    for chunk in iter(lambda: proc.stdout.read1(65536), b''):
        text = decoder.decode(chunk)
        sys.stdout.write(text)
        sys.stdout.flush()
        tail = (tail + text)[-20000:]
    code = proc.wait()
    if code:
        raise CommandError(code, tail)


def uv_venv(path, python):
    """Create a fresh virtualenv with uv, installing uv into the kernel first if needed."""
    if not shutil.which('uv'):
        sh(f'{shlex.quote(sys.executable)} -m pip install -q uv')
    shutil.rmtree(path, ignore_errors=True)
    sh(f'uv venv --python {python} {shlex.quote(str(path))}')


def require(path, cell):
    if not Path(path).exists():
        raise FileNotFoundError(f'{path} is missing; run cell {cell} first.')


def extract_zip(archive, dest, exts=IMAGE_EXTS):
    """Extract the images of a zip archive into dest, flattening folders into the file names."""
    with zipfile.ZipFile(archive) as zf:
        for member in sorted(zf.namelist()):
            path = Path(member)
            if path.suffix.lower() in exts and not path.name.startswith('.') and '__MACOSX' not in path.parts:
                (dest / '_'.join(path.parts)).write_bytes(zf.read(member))


def upload_files(kind):
    """Upload files through the browser into a new directory under OUTPUTS/uploads; zip archives are unpacked."""
    if not IN_COLAB:
        raise RuntimeError('Uploading needs Colab; set the path parameter instead.')
    from google.colab import files
    dest = OUTPUTS / 'uploads' / f'{kind}-{time.strftime("%Y%m%d-%H%M%S")}'
    dest.mkdir(parents=True)
    for name, data in files.upload().items():
        target = dest / Path(name).name
        target.write_bytes(data)
        Path(name).unlink(missing_ok=True)  # files.upload() also saves a copy in the working directory
        if target.suffix.lower() == '.zip':
            extract_zip(target, dest)
            target.unlink()
    return dest


VIEWER_PORT = 8765


def ensure_file_server():
    """Serve OUTPUTS over HTTP so Colab can proxy the HTML viewers and videos into the notebook."""
    def is_up():
        with socket.socket() as s:
            return s.connect_ex(('127.0.0.1', VIEWER_PORT)) == 0
    if is_up():
        return
    subprocess.Popen([sys.executable, '-m', 'http.server', str(VIEWER_PORT), '--directory', str(OUTPUTS)],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    for _ in range(100):
        time.sleep(0.1)
        if is_up():
            return
    raise RuntimeError('Could not start the local file server for the viewer.')


def served_path(path):
    return '/' + urllib.parse.quote(Path(os.path.abspath(path)).relative_to(OUTPUTS.resolve()).as_posix())


def list_files(directory):
    for path in sorted(Path(directory).rglob('*')):
        if path.is_file():
            print(f'  {path.relative_to(directory)!s:40s} {path.stat().st_size / 1e6:8.1f} MB')


def download(path):
    """Download a file, or a directory as a zip, through the browser."""
    path = Path(path)
    if path.is_dir():
        path = Path(shutil.make_archive(str(path), 'zip', root_dir=path))
    if IN_COLAB:
        from google.colab import files
        files.download(str(path))
    else:
        print('Saved at', path)




# Every script runs from the repository root, with absolute paths and one shared model cache.
GAE_ENV = env_with(ENVS / 'gae' / 'bin', HF_HOME=str(HF_HOME), HF_HUB_CACHE=str(HF_HOME / 'hub'),
                   HUGGINGFACE_HUB_CACHE=str(HF_HOME / 'hub'), TOKENIZERS_PARALLELISM='false',
                   PYTHONUNBUFFERED='1', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
                   PYTHONPATH=f'{REPO / "src"}:{REPO / "scripts" / "eval"}')


def fewer(n):
    """The count to retry with after running out of GPU memory: about 20% fewer, on the 16k+1 grid of the
    repository's app (..., 161, 129, 97, 81, 65, 49, 33, 17). None once 17 is reached."""
    if n <= 17:
        return None
    m = round((max(17, int(n * 0.8)) - 1) / 16) * 16 + 1
    return m if m < n else max(17, n - 16)


def video_frames(path):
    """Frame count of a video (0 if unknown)."""
    probe = 'import cv2, sys; print(int(cv2.VideoCapture(sys.argv[1]).get(cv2.CAP_PROP_FRAME_COUNT)))'
    out = subprocess.run([str(GAE_PY), '-c', probe, str(path)], capture_output=True, text=True).stdout.strip()
    return int(out) if out.isdigit() else 0


def run_gae(argv, show=None):
    """Run one GAE script from the repository root, reporting GPU use."""
    start = time.time()
    with GpuMonitor():
        sh(shlex.join(map(str, argv)), cwd=REPO, env=GAE_ENV, show=show)
    print(f'Finished in {time.time() - start:.0f} s')


def new_output(kind, name):
    return OUTPUTS / kind / f'{name}-{time.strftime("%Y%m%d-%H%M%S")}'


def media_source(path, kind, exts):
    """One file from a path, or from a browser upload."""
    if path:
        return Path(path)
    found = [p for p in upload_files(kind).iterdir() if p.suffix.lower() in exts]
    if not found:
        raise ValueError(f'Upload a file with one of these extensions: {", ".join(sorted(exts))}.')
    return found[0]


def prepare_image(path, crop=True):
    """Upright RGB copy of an image, centre-cropped to the model's 672x378 (16:9) frame unless crop=False.

    generate.py stretches its input to 672x378, so cropping first keeps proportions.
    """
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert('RGB')
    if crop:
        w, h = im.size
        if w / h > 672 / 378:
            nw = round(h * 672 / 378)
            im = im.crop(((w - nw) // 2, 0, (w - nw) // 2 + nw, h))
        else:
            nh = round(w * 378 / 672)
            im = im.crop((0, (h - nh) // 2, w, (h - nh) // 2 + nh))
    dest = OUTPUTS / 'inputs' / f'{Path(path).stem}-{time.strftime("%Y%m%d-%H%M%S")}.jpg'
    dest.parent.mkdir(parents=True, exist_ok=True)
    im.save(dest, quality=95)
    return dest


def describe_image(image):
    """A scene description for the prompt, written by the repository's Qwen2.5-VL captioner (~16 GB)."""
    result = OUTPUTS / 'inputs' / f'{Path(image).stem}-caption.txt'
    result.parent.mkdir(parents=True, exist_ok=True)
    script = ('import sys\nfrom scripts.demo.captioner import stream\ntext = ""\n'
              'for text in stream(sys.argv[1]):\n    pass\nopen(sys.argv[2], "w").write(text)\n')
    run_gae([GAE_PY, '-c', script, image, result])
    caption = result.read_text().strip()
    print('Prompt:', caption)
    return caption


# ---------------------------------------------------------------- Part A: image -> video
def run_i2v(image, prompt, name, poses=None, trajectory='forward', speed=0.06, views=81, steps=50, cfg=2.0,
            seed=42, pc_stride=4, progressive=True, on_oom='fewer views'):
    """generate.py on one image, along a recorded camera path (`poses` NPZ) or a synthetic `trajectory`.

    A recorded path gives one frame per camera (up to `views`), generated in one pass. A synthetic path has
    `views` frames, any number: beyond 81 it is generated 81 at a time (chunked rollout). If a pass does not fit in
    GPU memory, it is retried with fewer views per pass; a synthetic path keeps all its frames, a recorded one
    gets fewer.
    """
    require(GAE_PY, '1.1')
    require(CKPTS / 'flow_gae64.pt', '1.2')
    if poses is not None:
        cameras = len(np.load(poses)['c2w'])
        if views > cameras:
            print(f'The camera file has {cameras} cameras, so the video has {cameras} frames.')
            views = cameras
        chunk = views
    else:
        chunk = min(views, TRAINED_VIEWS)
    out = new_output('i2v', name)
    argv = [GAE_PY, REPO / 'scripts' / 'demo' / 'generate.py', '--image', Path(image).absolute(),
            '--prompt', prompt, '--output', out, '--ckpt-dir', CKPTS, '--flow-ckpt', CKPTS / 'flow_gae64.pt',
            '--codec-ckpt', CKPTS / 'gae_64.pt', '--sample-steps', steps, '--cfg-scale', cfg, '--seed', seed,
            '--pc-stride', pc_stride]
    if poses is not None:
        argv += ['--poses', Path(poses).absolute()]
        if progressive:
            argv.append('--dump-geometry')  # depth sidecar for the progressive point-cloud video
    else:
        argv += ['--free-rollout', '--trajectory', trajectory, '--speed', speed,
                 '--trajectory-reference-poses', DEFAULT_POSES]
    while True:
        counts = ['--num-views', chunk, '--total-views', chunk if poses is not None else views]
        label = f'{chunk} views' if chunk == views else f'{views} frames, {chunk} per pass (chunked rollout)'
        print(f'Generating {label}, {steps} steps.')
        try:
            run_gae(argv + counts, show=shlex.join(map(str, argv[:4] + counts)) + ' …')
            break
        except CommandError as error:
            if not (error.oom and on_oom == 'fewer views' and fewer(chunk)):
                raise
            chunk = fewer(chunk)
            shutil.rmtree(out, ignore_errors=True)
            kept = f'; the video keeps all {views} frames' if poses is None else ''
            print(f'\nOut of GPU memory; retrying with {chunk} views per pass{kept}.\n')
    scene = out / 'scannetpp'
    ply = scene / '000_pred_pointcloud.ply'
    if poses is not None and progressive and ply.is_file():
        try:  # the repository demo's cumulative point-cloud video
            run_gae([GAE_PY, REPO / 'scripts' / 'demo' / 'render_progressive_ply.py', ply,
                     '--output-prefix', scene / f'{name}_progressive', '--depth-edge-filter',
                     '--depth-edge-threshold', '0.05', '--point-budget', '0', '--point-size', '1',
                     '--render-voxel-size', '0.001', '--camera-back-offset', '0.2', '--hold-frames', '1'])
        except CommandError:
            print('The progressive point-cloud video could not be rendered (this is known to happen for scenes '
                  'with very distant depth); the other outputs are complete.')
    return out


def show_i2v(out):
    scene = Path(out) / 'scannetpp'
    print(f'Results in {out}')
    list_files(scene)
    show_videos([('Generated video', scene / '000_pred.mp4'), ('Decoded depth', scene / '000_depth.mp4')]
                + [('Progressive point cloud', p) for p in sorted(scene.glob('*_progressive.mp4'))])
    if (scene / '000_trajectory.png').is_file():
        from IPython.display import Image as IPImage
        display(IPImage(filename=str(scene / '000_trajectory.png'), width=640))
    if (scene / '000_pred_pointcloud.ply').is_file():
        plot_points(scene / '000_pred_pointcloud.ply', title='Generated scene (decoded geometry)')


# ---------------------------------------------------------------- Part B: text -> image
def run_t2i(prompts, num_images=3, guidance='ig', ig_scale=2.0, cfg_scale=2.0, steps=50, seed=0, point_cloud=True):
    require(GAE_PY, '1.1')
    require(CKPTS / 'flow_gae64.pt', '1.2')
    out = new_output('t2i', 'prompts')
    out.mkdir(parents=True)
    prompts_file = out / 'requested_prompts.txt'
    prompts_file.write_text('\n'.join(prompts) + '\n')
    run_gae([GAE_PY, REPO / 'scripts' / 'demo' / 'generate_t2i.py', '--ckpt-dir', CKPTS,
             '--flow-ckpt', CKPTS / 'flow_gae64.pt', '--codec-ckpt', CKPTS / 'gae_64.pt',
             '--prompts-file', prompts_file, '--num-images', num_images, '--guidance', guidance,
             '--ig-scale', ig_scale, '--cfg-scale', cfg_scale, '--sample-steps', steps, '--seed', seed,
             '--output', out, '--save-pointcloud' if point_cloud else '--no-pointcloud'])
    return out


def show_t2i(out, columns=3):
    import matplotlib.pyplot as plt
    images = sorted(p for p in Path(out).glob('*.png') if not p.stem.endswith('_depth'))
    print(f'{len(images)} image(s) in {out}')
    rows = []
    for image in images:
        depth = image.with_name(image.stem + '_depth.png')
        rows.append((image, depth if depth.is_file() else None))
    for start in range(0, len(rows), columns):
        batch = rows[start:start + columns]
        has_depth = any(d for _, d in batch)
        fig, axes = plt.subplots(2 if has_depth else 1, len(batch), squeeze=False,
                                 figsize=(5 * len(batch), 6 if has_depth else 3))
        for col, (image, depth) in enumerate(batch):
            for row, path in enumerate([image, depth] if has_depth else [image]):
                ax = axes[row][col]
                ax.axis('off')
                if path is not None:
                    ax.imshow(Image.open(path))
                    ax.set_title(path.stem[:48], fontsize=8)
        plt.tight_layout()
        plt.show()
    clouds = sorted(Path(out).glob('*_pointcloud.ply'))
    if clouds:
        plot_points(clouds[0], title=f'Point cloud of {clouds[0].stem.replace("_pointcloud", "")[:60]}')


# ---------------------------------------------------------------- Part C: reconstruction
def run_recon(media, max_frames=0, pc_stride=4, on_oom='fewer frames'):
    """reconstruct_vae.py on a video (every frame, or the first `max_frames`) or one image.

    All frames are encoded in one pass; if that does not fit in GPU memory, it is retried with fewer frames.
    """
    require(GAE_PY, '1.1')
    media = Path(media).absolute()
    is_video = media.suffix.lower() in VIDEO_EXTS
    out = new_output('recon', media.stem)
    argv = [GAE_PY, REPO / 'scripts' / 'demo' / 'reconstruct_vae.py', '--video' if is_video else '--image', media,
            '--hf-repo', HF_REPO, '--cache-dir', CKPTS, '--output', out, '--pc-stride', pc_stride]
    frames = max_frames
    while True:
        try:
            run_gae(argv + (['--max-frames', frames] if is_video else []))
            break
        except CommandError as error:
            if not (is_video and error.oom and on_oom == 'fewer frames'):
                raise
            total = video_frames(media)
            used = min(frames, total) if frames and total else (frames or total)
            if not fewer(used):
                raise
            frames = fewer(used)
            shutil.rmtree(out, ignore_errors=True)
            print(f'\nOut of GPU memory; retrying with the first {frames} frames.\n')
    case = out / media.stem
    shutil.copy(media, case / f'input{media.suffix.lower()}')  # so the original can be shown next to the result
    return case


def show_recon(case):
    case = Path(case)
    print(f'Results in {case}')
    list_files(case)
    original = next(case.glob('input.*'))
    if original.suffix.lower() in VIDEO_EXTS:
        show_videos([('Input', original), ('Reconstructed RGB', case / 'rgb_recon.mp4'),
                     ('Decoded depth', case / 'depth_recon.mp4')])
    else:
        import matplotlib.pyplot as plt
        panels = [('Input', original), ('Reconstructed RGB', case / 'rgb_recon.png'),
                  ('Decoded depth', case / 'depth_recon.png')]
        panels = [(t, p) for t, p in panels if p.is_file()]
        fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 3.2))
        for ax, (title, path) in zip(np.atleast_1d(axes), panels):
            ax.imshow(Image.open(path))
            ax.set_title(title)
            ax.axis('off')
        plt.tight_layout()
        plt.show()
    if (case / 'recon_pointcloud.ply').is_file():
        plot_points(case / 'recon_pointcloud.ply', title='Reconstructed point cloud')


# ---------------------------------------------------------------- point clouds and videos
def read_points(ply):
    """xyz and rgb in [0, 1] from an ASCII or binary PLY with x, y, z, red, green, blue vertices."""
    with open(ply, 'rb') as f:
        header = []
        while not header or header[-1] != 'end_header':
            header.append(f.readline().decode('ascii', 'replace').strip())
        fmt = next(line.split()[1] for line in header if line.startswith('format'))
        count = next(int(line.split()[2]) for line in header if line.startswith('element vertex'))
        props = [(line.split()[2], line.split()[1]) for line in header if line.startswith('property')]
        if fmt == 'ascii':
            import pandas as pd
            data = pd.read_csv(f, sep=r'\s+', header=None, nrows=count, engine='c').to_numpy()
            columns = {name: data[:, i] for i, (name, _) in enumerate(props)}
        else:
            types = {'float': 'f4', 'float32': 'f4', 'double': 'f8', 'float64': 'f8', 'uchar': 'u1', 'uint8': 'u1',
                     'char': 'i1', 'int8': 'i1', 'ushort': 'u2', 'short': 'i2', 'int': 'i4', 'uint': 'u4'}
            order = '<' if 'little' in fmt else '>'
            dtype = np.dtype([(name, order + types[kind]) for name, kind in props])
            data = np.frombuffer(f.read(count * dtype.itemsize), dtype=dtype, count=count)
            columns = {name: data[name] for name, _ in props}
    xyz = np.stack([columns[k] for k in ('x', 'y', 'z')], 1).astype(np.float32)
    rgb = np.stack([columns[k] for k in ('red', 'green', 'blue')], 1).astype(np.float32) / 255
    return xyz, rgb


def plot_points(ply, title='', max_points=80_000):
    """Interactive 3D view of a random subset of a point cloud (OpenCV camera frame: y down, z forward)."""
    import plotly.graph_objects as go
    xyz, rgb = read_points(ply)
    keep = np.arange(len(xyz))
    if len(keep) > max_points:
        keep = np.sort(np.random.default_rng(0).choice(len(xyz), max_points, replace=False))
    p, c = xyz[keep], (rgb[keep] * 255).astype(np.uint8)
    lo, hi = np.percentile(p, [1, 99], axis=0)
    margin = (hi - lo) * 0.1
    inside = np.all((p >= lo - margin) & (p <= hi + margin), axis=1)  # keep far outliers out of the view box
    p, c = p[inside], c[inside]
    hidden = dict(visible=False)
    fig = go.Figure(go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='markers', hoverinfo='skip',
                                 marker=dict(size=1.5, color=[f'rgb({r},{g},{b})' for r, g, b in c])))
    fig.update_layout(title=f'{title} ({len(p):,} of {len(xyz):,} points, drag to rotate)', height=600,
                      margin=dict(l=0, r=0, t=40, b=0), paper_bgcolor='#666666', font=dict(color='white'),
                      scene=dict(aspectmode='data', xaxis=hidden, yaxis=hidden, zaxis=hidden, bgcolor='#666666',
                                 camera=dict(up=dict(x=0, y=-1, z=0), eye=dict(x=0, y=-0.4, z=-1.8))))
    fig.show()


def show_videos(items):
    """Looping videos side by side: [(label, path), ...]; missing files are skipped."""
    items = [(label, Path(p)) for label, p in items if p is not None and Path(p).is_file()]
    if not items:
        return
    if not IN_COLAB:
        for label, path in items:
            print(f'{label}: {path}')
        return
    from google.colab.output import eval_js
    ensure_file_server()
    base = eval_js(f'google.colab.kernel.proxyPort({VIEWER_PORT})').rstrip('/')
    cells = ''.join(
        f'<figure style="margin:0;flex:1 1 320px"><video src="{base}{served_path(path)}" controls autoplay loop '
        f'muted playsinline style="width:100%"></video><figcaption style="font:13px sans-serif">'
        f'{html.escape(label)}</figcaption></figure>' for label, path in items)
    display(HTML(f'<div style="display:flex;gap:8px;flex-wrap:wrap">{cells}</div>'))


def download_results(out, everything=False):
    """Zip a result folder for download; by default without the large latent and input sidecars."""
    out = Path(out)
    heavy = ('_latents.npz', '_geom.npz')
    archive = out.with_name(out.name + ('-all' if everything else '') + '.zip')
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(out.rglob('*')):
            skip = not everything and (path.name.endswith(heavy) or '_input_scene' in path.parts)
            if path.is_file() and not skip:
                zf.write(path, path.relative_to(out))
    print(f'{archive.name}: {archive.stat().st_size / 1e6:.0f} MB')
    download(archive)


print('Helpers ready.')

## 1 · Install and download

In [ ]:
#@title 1.1 · Install the GAE environment with CUDA PyTorch (about 4–6 min)
TORCH_CUDA = 'auto'  #@param ["auto", "cu124", "cu121"]
#@markdown `auto` installs the CUDA 12.4 build of PyTorch 2.5.1 when the driver supports it, otherwise the CUDA 12.1 build. The cell ends with `CUDA OK` and an xFormers check on the GPU; it fails if either cannot use the GPU.

# The repository's pins (torch 2.5.1, Python 3.10) in an isolated env. The constraints match its provisioning
# script; transformers < 5.19 and diffusers < 0.41 are the newest releases that still import with torch 2.5.1.
CONSTRAINTS = '\n'.join(['torch==2.5.1', 'torchvision==0.20.1', 'xformers==0.0.28.post3', 'numpy<2',
                         'transformers>=4.51,<5.19', 'diffusers<0.41', 'depth-anything-3==0.1.1'])
tag = TORCH_CUDA if TORCH_CUDA != 'auto' else ('cu124' if DRIVER_CUDA >= (12, 4) else 'cu121')
marker = ENVS / 'gae' / '.ready'
if not marker.is_file() or marker.read_text() != tag:
    uv_venv(ENVS / 'gae', '3.10')
    constraints = ENVS / 'gae-constraints.txt'
    constraints.write_text(CONSTRAINTS + '\n')
    pip = f'uv pip install --python {GAE_PY}'
    sh(f'{pip} torch==2.5.1 torchvision==0.20.1 xformers==0.0.28.post3 '
       f'--index-url https://download.pytorch.org/whl/{tag}')
    sh(f'{pip} -c {constraints} -e {REPO}')
    marker.write_text(tag)
check_cuda(GAE_PY)
# GAE's video transformer uses xFormers attention when it is installed; make sure its CUDA kernels run.
XFORMERS_CHECK = ("import torch, xformers, xformers.ops as xo; "
                  "q = torch.randn(1, 1296, 8, 64, device='cuda', dtype=torch.bfloat16); "
                  "xo.memory_efficient_attention(q, q, q); "
                  "print('xFormers', xformers.__version__, 'memory-efficient attention OK on the GPU')")
sh(f'{GAE_PY} -c {shlex.quote(XFORMERS_CHECK)}')
sh(f'{GAE_PY} -c "import gae, transformers, diffusers; '
   f'print(\'gae OK; transformers\', transformers.__version__, \'diffusers\', diffusers.__version__)"', cwd=REPO)

In [ ]:
#@title 1.2 · Download the weights (about 13 GB)
DOWNLOAD_CAPTION_MODEL = False  #@param {type:"boolean"}
#@markdown The GAE-64 weights (5.6 GB) go to `ckpts/` in the repository; DA3-GIANT (5.4 GB) and Qwen3-0.6B (1.5 GB) go to one shared Hugging Face cache. `DOWNLOAD_CAPTION_MODEL` also fetches Qwen2.5-VL-7B (about 16 GB) for automatic prompts in A.2; it is otherwise downloaded on first use.
require(GAE_PY, '1.1')
sh(f'{GAE_PY} scripts/demo/download_checkpoints.py --size 64 --out-dir {CKPTS}', cwd=REPO, env=GAE_ENV)
models = HUB_MODELS + ([CAPTION_MODEL] if DOWNLOAD_CAPTION_MODEL else [])
sh(f'{GAE_PY} -c "import sys; from huggingface_hub import snapshot_download; '
   f'[print(snapshot_download(r)) for r in sys.argv[1:]]" ' + ' '.join(models), env=GAE_ENV)
list_files(CKPTS)

## Part A · Image → camera-controlled video + 3D

GAE encodes the image as a clean reference, then a flow transformer generates the following views along a metric camera path (Plücker rays), conditioned on the prompt. The decoded latents give the video **and** its depth, from which the point cloud is built.

The released recipe is 81 views and 50 sampling steps at 672×378, which takes a while on one GPU (the authors' interactive demo spreads it over eight). `fast preview` (17 views, 25 steps) is a quick first check. With `QUALITY = custom`, `VIEWS` can be any number: a recorded camera path gives one frame per camera, and a synthetic path longer than 81 frames is generated 81 frames at a time (the model's training clip length) with chunked rollout.

In [ ]:
#@title A.1 · Bundled example scene (recorded camera path)
SCENE = 'forest_lake_trail'  #@param ["autumn_waterfall", "bedroom", "city_street", "forest_lake_trail", "hillside_car", "office_desk"]
QUALITY = 'full (81 views, 50 steps)'  #@param ["full (81 views, 50 steps)", "fast preview (17 views, 25 steps)", "custom"]
VIEWS = 81  #@param {type:"integer"}
SAMPLE_STEPS = 50  #@param {type:"integer"}
CFG_SCALE = 2.0  #@param {type:"number"}
SEED = 42  #@param {type:"integer"}
PC_STRIDE = 4  #@param {type:"integer"}
PROGRESSIVE_VIDEO = True  #@param {type:"boolean"}
ON_OOM = 'fewer views'  #@param ["fewer views", "stop"]
#@markdown * `VIEWS` and `SAMPLE_STEPS` apply with `QUALITY = custom`. Each example ships its prompt and a recorded camera path (`<scene>_poses.npz`) of 81 cameras, so its video has at most 81 frames; for longer videos use A.2 with a synthetic path.
#@markdown * `PC_STRIDE` thins the point cloud (1 keeps every ray: about 83k points per view). `PROGRESSIVE_VIDEO` renders the repository's cumulative point-cloud video.
#@markdown * If a run does not fit in GPU memory, `fewer views` retries with about 20% fewer views each time (81 → 65 → 49 → 33 → 17).

views, steps = {'full (81 views, 50 steps)': (81, 50), 'fast preview (17 views, 25 steps)': (17, 25)}.get(QUALITY, (VIEWS, SAMPLE_STEPS))
scene_image = REPO / 'examples' / 'scenes' / f'{SCENE}.jpg'
prompt = scene_image.with_suffix('.txt').read_text().strip()
print('Prompt:', prompt)
LAST_I2V = run_i2v(scene_image, prompt, SCENE, poses=scene_image.with_name(f'{SCENE}_poses.npz'), views=views,
                   steps=steps, cfg=CFG_SCALE, seed=SEED, pc_stride=PC_STRIDE, progressive=PROGRESSIVE_VIDEO,
                   on_oom=ON_OOM)
show_i2v(LAST_I2V)

In [ ]:
#@title A.2 · Your own image
IMAGE_PATH = ''  #@param {type:"string"}
PROMPT = ''  #@param {type:"string"}
AUTO_PROMPT = False  #@param {type:"boolean"}
CROP_TO_16_9 = True  #@param {type:"boolean"}
CAMERA = 'default path (forest trail)'  #@param ["default path (forest trail)", "forward", "backward", "turn_left", "turn_right", "my camera file"]
CAMERA_NPZ = ''  #@param {type:"string"}
SPEED = 0.06  #@param {type:"number"}
QUALITY = 'full (81 views, 50 steps)'  #@param ["full (81 views, 50 steps)", "fast preview (17 views, 25 steps)", "custom"]
VIEWS = 81  #@param {type:"integer"}
SAMPLE_STEPS = 50  #@param {type:"integer"}
CFG_SCALE = 2.0  #@param {type:"number"}
SEED = 42  #@param {type:"integer"}
PC_STRIDE = 4  #@param {type:"integer"}
ON_OOM = 'fewer views'  #@param ["fewer views", "stop"]
#@markdown * Leave `IMAGE_PATH` empty to upload an image (JPG/PNG/WebP), or give a path (for Google Drive, mount Drive first with the folder icon on the left). The model works at 672×378; `CROP_TO_16_9` centre-crops instead of stretching.
#@markdown * Describe the scene in `PROMPT` (see `examples/scenes/*.txt` for the style), or tick `AUTO_PROMPT` to have Qwen2.5-VL-7B write it (about 16 GB, downloaded on first use).
#@markdown * Camera:
#@markdown   * `default path` follows the forest-trail recording, like the repository's app.
#@markdown   * `forward`, `backward`, `turn_left` and `turn_right` are synthetic paths scaled to that recording; `SPEED` is the step per frame. They can be any length (`QUALITY = custom`, `VIEWS` = frames): beyond 81 frames, or on out-of-memory, the clip is generated in chunks and keeps every frame. They give no depth video or progressive video.
#@markdown   * `my camera file` uses `CAMERA_NPZ`: `c2w` `[N,4,4]` OpenCV camera-to-world in metres and `K` `[N,3,3]` in pixels of the 672×378 frame. The video has one frame per camera (up to `VIEWS`), generated in one pass.

source = media_source(IMAGE_PATH, 'image', IMAGE_EXTS)
image = prepare_image(source, crop=CROP_TO_16_9)
prompt = PROMPT.strip() or (describe_image(image) if AUTO_PROMPT else '')
if not prompt:
    raise ValueError('Write a PROMPT describing the scene, or tick AUTO_PROMPT.')
poses = {'default path (forest trail)': DEFAULT_POSES, 'my camera file': CAMERA_NPZ or None}.get(CAMERA)
if CAMERA == 'my camera file' and poses is None:
    raise ValueError('Set CAMERA_NPZ to your camera file.')
views, steps = {'full (81 views, 50 steps)': (81, 50), 'fast preview (17 views, 25 steps)': (17, 25)}.get(QUALITY, (VIEWS, SAMPLE_STEPS))
LAST_I2V = run_i2v(image, prompt, source.stem, poses=poses, trajectory=CAMERA, speed=SPEED, views=views,
                   steps=steps, cfg=CFG_SCALE, seed=SEED, pc_stride=PC_STRIDE, on_oom=ON_OOM)
show_i2v(LAST_I2V)

In [ ]:
#@title A.3 · Download the latest video result
EVERYTHING = False  #@param {type:"boolean"}
#@markdown By default the zip leaves out the large latent and geometry sidecars (`*_latents.npz` is about 1 GB).
download_results(LAST_I2V, everything=EVERYTHING)

## Part B · Text → image + depth + point cloud

The same flow transformer, with a single view and no camera, generates an image that the codec decodes into RGB and DPT depth. Guidance defaults to the repository's internal guidance (`ig`).

In [ ]:
#@title B.1 · Text → image
EXAMPLE_PROMPT = "A Day of the Dead sugar-skull cup with blue eyes and a pink flower."  #@param ["all bundled prompts", "A Day of the Dead sugar-skull cup with blue eyes and a pink flower.", "A Bohemian bedroom with a wooden bed and red patterned textiles.", "A smiling fox wearing a top hat and a waistcoat in a cloudy sky.", "A serene bedroom with a white comforter and orange pillows.", "A painterly still life of fruit in an ornate blue bowl.", "A humanoid robot with reflective armor and glowing green eyes in a misty forest.", "A weathered metallic skull half-buried in snow, with glowing red eyes.", "A close-up sculpture of Medusa in dark stone with coiled snakes.", "An antique French gilt-and-silver heart-shaped reliquary with ornate floral engraving, displayed on a wooden table under warm studio lighting.", "A charming anthropomorphic white-and-tan hamster wearing polished medieval plate armor, standing among vivid pink flowers.", "A detailed humanoid robot with reflective black-and-silver armor and glowing green eyes, standing in a misty forest at sunset.", "A peaceful oil painting of a young boy seated beneath a sun-dappled tree with two small dogs in a meadow.", "A lone hiker stands on moss-covered boulders overlooking a calm forest lake in a dark, serene landscape.", "A comic-style illustration of a large, ferocious green-and-black bear walking through a dense forest.", "A warm Moroccan-inspired living room with a rust-red sofa, turquoise cushions, carved wooden table, patterned rug, and blue vase.", "A slice of lemon meringue pie decorated with piped cream, elegantly presented on a blue-and-white ceramic plate.", "A weathered metallic skull half-buried in snow, with glowing red eyes and a fiery orange energy core in its temple.", "A detailed close-up sculpture of Medusa, rendered in dark green stone with intricately coiled snakes surrounding her face."]
MY_PROMPTS = ''  #@param {type:"string"}
NUM_IMAGES = 3  #@param {type:"integer"}
GUIDANCE = 'ig'  #@param ["ig", "cfg", "cfg_ig", "none"]
IG_SCALE = 2.0  #@param {type:"number"}
CFG_SCALE = 2.0  #@param {type:"number"}
SAMPLE_STEPS = 50  #@param {type:"integer"}
SEED = 0  #@param {type:"integer"}
POINT_CLOUD = True  #@param {type:"boolean"}
#@markdown * `MY_PROMPTS` overrides the example; separate several prompts with `;;`. `NUM_IMAGES` is per prompt.
#@markdown * `IG_SCALE` applies to `ig`/`cfg_ig`, `CFG_SCALE` to `cfg`/`cfg_ig`. `POINT_CLOUD` also decodes depth and a point cloud.

if MY_PROMPTS.strip():
    prompts = [p.strip() for p in MY_PROMPTS.split(';;') if p.strip()]
else:
    bundled = [l.strip() for l in (REPO / 'examples' / 't2i_prompts.txt').read_text().splitlines()
               if l.strip() and not l.startswith('#')]
    prompts = bundled if EXAMPLE_PROMPT == 'all bundled prompts' else [EXAMPLE_PROMPT]
LAST_T2I = run_t2i(prompts, num_images=NUM_IMAGES, guidance=GUIDANCE, ig_scale=IG_SCALE, cfg_scale=CFG_SCALE,
                   steps=SAMPLE_STEPS, seed=SEED, point_cloud=POINT_CLOUD)
show_t2i(LAST_T2I)

In [ ]:
#@title B.2 · Download the latest images (zip)
download_results(LAST_T2I)

## Part C · Codec reconstruction (RGB + depth + point cloud)

This tests the codec alone: frames are encoded into GAE's latent and decoded back into RGB and DPT depth, from which a point cloud is built. No generation is involved. Frames are resized to 672×378, the resolution the released models use.

In [ ]:
#@title C.1 · Reconstruct a video or an image
SOURCE = 'airport_luggage_vehicle'  #@param ["airport_luggage_vehicle", "autumn_waterfall", "bathroom_hallway", "historic_courtyard", "office_shelf", "offroad_vehicle", "my video or image"]
MEDIA_PATH = ''  #@param {type:"string"}
MAX_FRAMES = 0  #@param {type:"integer"}
PC_STRIDE = 4  #@param {type:"integer"}
ON_OOM = 'fewer frames'  #@param ["fewer frames", "stop"]
#@markdown * With `my video or image`, leave `MEDIA_PATH` empty to upload (MP4/MOV/WebM… or JPG/PNG/WebP), or give a path.
#@markdown * `MAX_FRAMES = 0` reconstructs every frame of the video, in one pass; a positive value keeps only the first `MAX_FRAMES`. The codec was trained on 81-frame clips (the bundled videos have 81 frames).
#@markdown * If a pass does not fit in GPU memory, `fewer frames` retries with about 20% fewer frames each time (the first frames of the video are kept).

if SOURCE == 'my video or image':
    media = media_source(MEDIA_PATH, 'recon', VIDEO_EXTS | IMAGE_EXTS)
else:
    media = REPO / 'examples' / 'recon_videos' / f'{SOURCE}.mp4'
LAST_RECON = run_recon(media, max_frames=MAX_FRAMES, pc_stride=PC_STRIDE, on_oom=ON_OOM)
show_recon(LAST_RECON)

In [ ]:
#@title C.2 · Download the latest reconstruction (zip)
download_results(LAST_RECON)

## Keep your results

Everything is written under `/content/outputs` (`i2v/`, `t2i/`, `recon/`, one new folder per run), which disappears when the Colab runtime is recycled. Download results with cells A.3, B.2 and C.2, or copy them all to Google Drive:

In [ ]:
#@title Copy all results to Google Drive
DRIVE_FOLDER = 'GAE_outputs'  #@param {type:"string"}
from google.colab import drive
drive.mount('/content/drive')
target = Path('/content/drive/MyDrive') / DRIVE_FOLDER
shutil.copytree(OUTPUTS, target, dirs_exist_ok=True, ignore=shutil.ignore_patterns('uploads', '*_latents.npz'))
print('Copied to', target)

## Troubleshooting

| Symptom | What to do |
| --- | --- |
| `CUDA OK` is missing, or a run prints *barely used the GPU* | Choose *Runtime → Change runtime type → A100 GPU*, then *Runtime → Disconnect and delete runtime* and run again from 0.1. On an older driver, rerun 1.1 with `TORCH_CUDA = cu121`. |
| `CUDA out of memory` | Part A and Part C retry with fewer views or frames by themselves (`ON_OOM`). With `stop`, use `fast preview` or fewer `VIEWS`/`MAX_FRAMES`. |
| A run used fewer views than requested | The full count did not fit in GPU memory; the output names the count that was used. On synthetic camera paths the full clip is kept and generated in chunks instead. A recorded camera path also caps the video at its number of cameras. |
| Exit code `-9` / `137`, or the runtime restarts | Out of system RAM while loading the models. Use an A100 or High-RAM runtime. |
| `NameError` after a runtime restart | Rerun cells 0.1–0.3. The environment and weights stay on disk until the runtime is recycled. |
| An install or download cell failed halfway | Run it again; installs start from scratch and downloads resume. |
| Distorted results from your own image | Keep `CROP_TO_16_9` on, and write a prompt that describes what is visible. |
| A warning that `gsplat` is required for rendering 3DGS | Harmless: it comes from the Depth Anything 3 package, whose Gaussian-splat renderer GAE does not use. |
| The progressive point-cloud video is missing | It needs a recorded camera path and can fail for scenes with very distant depth; the other outputs are unaffected. |

More details: [repository README](https://github.com/Cortex78/GAE-GeometricAutoEncoder#readme) · [examples](https://github.com/Cortex78/GAE-GeometricAutoEncoder/blob/main/examples/README.md)